In [1]:
print("prova")

prova


In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1" # Forza l'errore a uscire subito

In [2]:
import zipfile
import pandas as pd
import pydicom
import numpy as np
import cv2
from PIL import Image
from io import BytesIO

# --- CONFIGURAZIONE ---
ZIP_PATH = '/media/andy/Samsung/vinbigdata-chest-xray-abnormalities-detection.zip' 
CSV_PATH = '/home/andy/Documenti/Tesi-Magistrale/data/train.csv'        
OUTPUT_DIR = '/home/andy/Documenti/Tesi-Magistrale/data/RX_super' 
MODELLO_EDSR_PATH = '/home/andy/Documenti/Tesi-Magistrale/models_saved/EDSR_x2.pb'

def run_processor():
    # 1. Check cartelle
    os.makedirs(os.path.join(OUTPUT_DIR, 'sani'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'malati'), exist_ok=True)

    # 2. Caricamento e Bilanciamento
    print("Analisi dati...")
    df = pd.read_csv(CSV_PATH).drop_duplicates(subset=['image_id'])
    
    df_sani = df[df['class_name'] == 'No finding'].sample(n=2000, random_state=42)
    
    classi_oncologiche = ['Nodule/Mass', 'Lung Opacity', 'Pleural effusion']
    df_m = df[df['class_name'].isin(classi_oncologiche)]
    if len(df_m) < 2000:
        extra = df[(df['class_name'] != 'No finding') & (~df['class_name'].isin(classi_oncologiche))]
        df_m = pd.concat([df_m, extra.sample(n=2000-len(df_m), random_state=42)])
    df_malati = df_m.sample(n=2000, random_state=42)

    mappa_file = {f"train/{r['image_id']}.dicom": 'sani' for _, r in df_sani.iterrows()}
    mappa_file.update({f"train/{r['image_id']}.dicom": 'malati' for _, r in df_malati.iterrows()})
    
    lista_target = set(mappa_file.keys())

    # 3. Inizializzazione SR (Senza Lightning per ora)
    sr = cv2.dnn_superres.DnnSuperResImpl_create()
    sr.readModel(MODELLO_EDSR_PATH)
    sr.setModel("edsr", 2)
    
    # PROVA A DISABILITARE CUDA SE CRASHA ANCORA: 
    # A volte i driver NVIDIA su Ubuntu hanno memory leak con file enormi.
    # sr.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
    # sr.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)

    print(f"Inizio elaborazione. Obiettivo: {len(lista_target)} immagini.")
    
    # 4. Loop di estrazione sicuro
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            for percorso_zip in zf.namelist():
                nome = percorso_zip.strip()
                if nome in lista_target:
                    classe = mappa_file[nome]
                    out_path = os.path.join(OUTPUT_DIR, classe, os.path.basename(nome).replace('.dicom', '.jp2'))

                    if os.path.exists(out_path):
                        continue

                    # Processo singolo
                    with zf.open(percorso_zip) as f:
                        dcm = pydicom.dcmread(BytesIO(f.read()))
                        img = dcm.pixel_array.astype(float)
                        img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-7) * 255.0
                        img = img.astype(np.uint8)
                        if getattr(dcm, 'PhotometricInterpretation', '') == "MONOCHROME1":
                            img = 255 - img
                        
                        img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
                        img_sr = sr.upsample(img_bgr)
                        
                        img_pil = Image.fromarray(cv2.cvtColor(img_sr, cv2.COLOR_BGR2RGB))
                        img_pil.save(out_path, format='JPEG2000', quality_mode='dB', quality_vals=[0])
                        
                        print(f"OK: {os.path.basename(out_path)}")
    except Exception as e:
        print(f"Crash evitato, errore catturato: {e}")

if __name__ == "__main__":
    run_processor()

Analisi dati...
Inizio elaborazione. Obiettivo: 4000 immagini.


: 